# 29 · DINOv2 frozen heads: baseline, backbone comparison, calibration and selective prediction

Design log
- The accepted plan names DINOv2 as the principal anchor. ResNet-50 frozen, used in notebooks 26 to 28, is a control.
- DINOv2-small (dinov2_vits14, 384-dimensional CLS features, frozen) with base-rate initialised heads, seed 0. The Hugging Face snapshot is pinned on first use in features/dinov2_snapshot.json.
- Same learning-rate grid, 400-epoch cap and early stopping as notebook 27. Runs are written under runs/foundation_dinov2_prior_init/.
- The run per head is selected by validation loss, as for ResNet-50 in notebook 27. The backbone comparison uses the calibration partition, which neither backbone used for selection. DINOv2 minus ResNet-50 differences carry paired participant-group bootstrap intervals, 1000 replicates. A backbone chosen from this comparison is a recorded development decision; the sealed test remains the final evaluation.
- Calibration and selective prediction follow notebook 28: calibrators fitted on the calibration partition and compared on validation.
- Validation was used for early stopping and learning-rate selection, so absolute validation figures are slightly optimistic. Test records are not read.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Vision packages for DINOv2

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "accelerate>=1.0,<2",
                "huggingface-hub>=0.27,<1", "safetensors>=0.4"], check=True)
import transformers
print("transformers", transformers.__version__)

## 2. Frequency baseline, ResNet-50 reference and helpers

In [ ]:
import numpy as np, pandas as pd
from dataclasses import replace
from oncoplate.pipeline import frequency_baseline, load_study, spec_from_cfg, prediction_arrays
from oncoplate.training import train_predictor, predict_run
from oncoplate.calibration import calibration_metrics, probabilities, fit_temperature, fit_sigmoid
from oncoplate.statistics import analysis_weights, foundation_group_intervals
from oncoplate.selection import choose_threshold

HEADS = ("independent", "joint")
STUDY = {h: load_study(cfg, h, stage_images=True) for h in HEADS}
PREVALENCE = {h: frequency_baseline(cfg, h, "validation")["probabilities"][0] for h in HEADS}
# read_table loads every column as text; numeric types are needed to sort by loss.
RES_GRID = pd.read_csv(p["reports"] / "foundation_prior_init_grid.csv")
RES_CHOSEN = RES_GRID.sort_values(["head", "best_validation_bce", "lr"]).groupby("head").head(1).set_index("head")

def evaluate(prob, y, mask, ids, head):
    m = calibration_metrics(prob, y, mask)
    sub = STUDY[head][0].set_index("record_id").loc[ids].reset_index()
    w = analysis_weights(sub, {"meal": 1.0})
    rows = np.arange(len(prob)); j = prob.argmax(1)
    observed = mask[rows, j] > 0; den = w[observed].sum()
    top1 = float(np.sum(w[observed] * y[rows, j][observed]) / den) if den else float("nan")
    return {"log_loss": m["log_loss"], "ece": m["ece"], "top1_endorsement": top1}

def frequency_on(ids, y, mask, head):
    # Scored on exactly the records, targets and masks the image model is scored on.
    return evaluate(np.broadcast_to(PREVALENCE[head], y.shape).copy(), y, mask, ids, head)

def group_parts(prob, y, mask, ids, head):
    # Per-participant numerator and denominator of the weighted top-1 endorsement in evaluate().
    sub = STUDY[head][0].set_index("record_id").loc[ids].reset_index()
    w = analysis_weights(sub, {"meal": 1.0}); g = sub["group_id"].to_numpy()
    rows = np.arange(len(prob)); j = prob.argmax(1); observed = mask[rows, j] > 0
    num = pd.Series(np.where(observed, w * y[rows, j], 0.0)).groupby(g).sum()
    den = pd.Series(np.where(observed, w, 0.0)).groupby(g).sum()
    return num, den

def paired_difference(head, prob_a, prob_b, y, mask, ids, B=1000, seed=0):
    # Resamples participant groups; returns (log-loss, top-1 endorsement) of b minus a per draw.
    groups = STUDY[head][0].set_index("record_id").loc[ids, "group_id"].to_numpy()
    uniq = np.unique(groups); members = [np.flatnonzero(groups == g) for g in uniq]
    na, da = group_parts(prob_a, y, mask, ids, head); nb_, db = group_parts(prob_b, y, mask, ids, head)
    rng = np.random.default_rng(seed); out = []
    for _ in range(B):
        k = rng.integers(0, len(uniq), len(uniq)); pick = np.concatenate([members[i] for i in k]); g = uniq[k]
        ll = calibration_metrics(prob_b[pick], y[pick], mask[pick])["log_loss"] - calibration_metrics(prob_a[pick], y[pick], mask[pick])["log_loss"]
        t1 = nb_.loc[g].sum() / db.loc[g].sum() - na.loc[g].sum() / da.loc[g].sum()
        out.append((ll, t1))
    return np.asarray(out)

RES_CHOSEN[["lr", "epochs_completed", "best_validation_bce"]]

## 3. DINOv2 learning-rate grid with base-rate initialisation

In [ ]:
LEARNING_RATES = (0.001, 0.003, 0.01)
EPOCHS = 400
OUT = p["runs"] / "foundation_dinov2_prior_init"
rows = []
for head in HEADS:
    records, targets = STUDY[head]
    base = spec_from_cfg(cfg, "dinov2_vits14", "frozen", head, 0)
    for lr in LEARNING_RATES:
        spec = replace(base, lr=lr, epochs=EPOCHS, bias_init="prior")
        run_dir = OUT / f"{spec.run_id}_prior_lr{lr:g}_e{EPOCHS}"
        train_predictor(records, targets, spec, run_dir, p["features"])
        done = read_json(run_dir / "complete.json")
        rows.append({"head": head, "lr": lr, "run_dir": str(run_dir),
                     "epochs_completed": int(done["epochs_completed"]),
                     "early_stopped": int(done["epochs_completed"]) < EPOCHS,
                     "best_validation_bce": float(done["best_validation_loss"])})
grid = pd.DataFrame(rows)
write_table(p["reports"] / "foundation_dinov2_grid.csv", grid)
grid

## 4. Validation comparison on identical records

In [ ]:
table = []
for head in HEADS:
    ref = prediction_arrays(Path(RES_CHOSEN.loc[head, "run_dir"]) / "validation_predictions.npz")
    ids, y, mask = ref["ids"], ref["y"], ref["mask"]
    freq_row = frequency_on(ids, y, mask, head)
    saved = read_json(p["reports"] / f"frequency_{head}_validation_metrics.json")["log_loss"]
    assert np.isclose(freq_row["log_loss"], saved, rtol=1e-5), "Frequency baseline does not reproduce notebook 05"
    table.append({"head": head, "model": "frequency baseline (no image)", **freq_row})
    table.append({"head": head, "model": f"resnet50, lr {RES_CHOSEN.loc[head, 'lr']:g} (notebook 27)",
                  **evaluate(probabilities(ref["logits"]), y, mask, ids, head)})
    for r in grid[grid["head"] == head].itertuples():
        z = prediction_arrays(Path(r.run_dir) / "validation_predictions.npz")
        assert list(z["ids"]) == list(ids), "Validation record sets differ"
        table.append({"head": head, "model": f"dinov2, lr {r.lr:g}, stopped at {r.epochs_completed} of {EPOCHS}",
                      **evaluate(probabilities(z["logits"]), y, mask, ids, head)})
validation = pd.DataFrame(table)
write_table(p["reports"] / "foundation_dinov2_validation.csv", validation)
validation

## 5. Backbone comparison on the calibration partition

In [ ]:
DINO_CHOSEN = grid.sort_values(["head", "best_validation_bce", "lr"]).groupby("head").head(1).set_index("head")
PRED, comparison, gap = {}, [], []
for head in HEADS:
    records, targets = STUDY[head]; part = records[records.split.eq("calibration")]
    runs = {"resnet50": Path(RES_CHOSEN.loc[head, "run_dir"]), "dinov2": Path(DINO_CHOSEN.loc[head, "run_dir"])}
    cal = {k: predict_run(run, part, targets, p["features"]) for k, run in runs.items()}
    ids, y, mask = cal["dinov2"]["ids"], cal["dinov2"]["y"], cal["dinov2"]["mask"]
    assert list(cal["resnet50"]["ids"]) == list(ids), "Calibration record sets differ"
    PRED[head] = {"run": runs["dinov2"], "validation": prediction_arrays(runs["dinov2"] / "validation_predictions.npz"),
                  "calibration": cal["dinov2"]}
    prob = {k: probabilities(cal[k]["logits"]) for k in runs}
    scores = {k: evaluate(prob[k], y, mask, ids, head) for k in runs}
    comparison += [{"head": head, "model": "frequency baseline (no image)", **frequency_on(ids, y, mask, head)}] + \
                  [{"head": head, "model": f"{k}, base-rate head", **scores[k]} for k in runs]
    draws = paired_difference(head, prob["resnet50"], prob["dinov2"], y, mask, ids)
    for col, metric in enumerate(("log_loss", "top1_endorsement")):
        lo, hi = np.quantile(draws[:, col], [0.025, 0.975])
        gap.append({"head": head, "metric": metric, "dinov2_minus_resnet50": scores["dinov2"][metric] - scores["resnet50"][metric],
                    "ci95_low": float(lo), "ci95_high": float(hi), "ci_excludes_zero": bool(lo > 0 or hi < 0)})
comparison = pd.DataFrame(comparison); gap = pd.DataFrame(gap)
write_table(p["reports"] / "foundation_dinov2_calibration_partition.csv", comparison)
write_table(p["reports"] / "foundation_dinov2_backbone_gap.csv", gap)
display(comparison)
gap

## 6. Calibrators fitted on the calibration partition, compared on validation

In [ ]:
CAL, rows = {}, []
for head, d in PRED.items():
    c, v = d["calibration"], d["validation"]
    CAL[head] = {"identity": {"kind": "identity"},
                 "temperature": fit_temperature(c["logits"], c["y"], c["mask"]),
                 "sigmoid": fit_sigmoid(c["logits"], c["y"], c["mask"])}
    write_json(d["run"] / "calibrators.json", CAL[head])
    rows.append({"head": head, "calibrator": "frequency baseline (no image)", **frequency_on(v["ids"], v["y"], v["mask"], head)})
    for name, calib in CAL[head].items():
        rows.append({"head": head, "calibrator": name,
                     **evaluate(probabilities(v["logits"], calib), v["y"], v["mask"], v["ids"], head)})
calibration_table = pd.DataFrame(rows)
write_table(p["reports"] / "foundation_dinov2_calibration_validation.csv", calibration_table)
print("fitted temperature:", {h: round(CAL[h]["temperature"]["temperature"], 4) for h in HEADS})
calibration_table

## 7. Participant-group intervals for the calibration changes

In [ ]:
def paired_group_bootstrap(head, probs, y, mask, ids, B=1000, seed=0):
    # Resamples participant groups; every calibrator is scored on the same draw.
    groups = STUDY[head][0].set_index("record_id").loc[ids, "group_id"].to_numpy()
    members = [np.flatnonzero(groups == g) for g in np.unique(groups)]
    rng = np.random.default_rng(seed); draws = []
    for _ in range(B):
        pick = np.concatenate([members[i] for i in rng.integers(0, len(members), len(members))])
        m = {k: calibration_metrics(v[pick], y[pick], mask[pick]) for k, v in probs.items()}
        draws.append({f"{k}_{s}": m[k][s] - m["identity"][s] for k in probs if k != "identity" for s in ("log_loss", "ece")})
    return pd.DataFrame(draws)

rows = []
for head, d in PRED.items():
    v = d["validation"]
    probs = {k: probabilities(v["logits"], c) for k, c in CAL[head].items()}
    point = {k: calibration_metrics(pr, v["y"], v["mask"]) for k, pr in probs.items()}
    boot = paired_group_bootstrap(head, probs, v["y"], v["mask"], v["ids"])
    for k in ("temperature", "sigmoid"):
        for s in ("log_loss", "ece"):
            lo, hi = np.quantile(boot[f"{k}_{s}"], [0.025, 0.975])
            rows.append({"head": head, "calibrator": k, "metric": s,
                         "change_vs_uncalibrated": point[k][s] - point["identity"][s],
                         "ci95_low": float(lo), "ci95_high": float(hi), "ci_excludes_zero": bool(lo > 0 or hi < 0)})
gains = pd.DataFrame(rows)
write_table(p["reports"] / "foundation_dinov2_calibration_gains.csv", gains)
gains

## 8. Selective prediction on validation

In [ ]:
TARGETS = (1.0, 0.9, 0.8, 0.7, 0.6, 0.5)
rows = []
for head, d in PRED.items():
    v = d["validation"]; n = len(v["ids"]); idx = np.arange(n)
    sub = STUDY[head][0].set_index("record_id").loc[v["ids"]].reset_index()
    w = analysis_weights(sub, {"meal": 1.0})
    for name in ("identity", "temperature", "sigmoid"):
        prob = probabilities(v["logits"], CAL[head][name]); j = prob.argmax(1); conf = prob.max(1)
        observed = v["mask"][idx, j] > 0; agreement = v["y"][idx, j]
        for target in TARGETS:
            thr = choose_threshold(conf, np.ones(n, bool), target, w)
            frame = pd.DataFrame({"record_id": v["ids"], "group_id": sub["group_id"].to_numpy(),
                                  "accepted": conf >= thr["threshold"], "observed": observed, "agreement": agreement})
            est, _ = foundation_group_intervals(frame, B=1000)
            dis = est["metrics"]["panel_disagreement_among_assessable_answers"]
            rows.append({"head": head, "calibrator": name, "target_coverage": target,
                         "coverage": est["metrics"]["answer_coverage_all_records"]["estimate"],
                         "panel_disagreement": dis["estimate"], "ci95_low": dis["ci95"][0], "ci95_high": dis["ci95"][1]})
selective = pd.DataFrame(rows)
write_table(p["reports"] / "foundation_dinov2_selective_validation.csv", selective)
selective

## 9. Summary

In [ ]:
def pick(frame, value_col, **keys):
    sel = np.logical_and.reduce([np.isclose(frame[k], v) if isinstance(v, float) else frame[k].eq(v)
                                 for k, v in keys.items()])
    r = frame.loc[sel].iloc[0]
    return [round(float(r[value_col]), 4), round(float(r["ci95_low"]), 4), round(float(r["ci95_high"]), 4)]

verdict = {}
for head in HEADS:
    c = comparison[comparison["head"] == head].set_index("model")
    verdict[head] = {
        "dinov2_run": PRED[head]["run"].name,
        "calibration_partition_log_loss": {m: round(float(c.loc[m, "log_loss"]), 4) for m in c.index},
        "calibration_partition_top1": {m: round(float(c.loc[m, "top1_endorsement"]), 3) for m in c.index},
        "dinov2_minus_resnet50_log_loss": pick(gap, "dinov2_minus_resnet50", head=head, metric="log_loss"),
        "dinov2_minus_resnet50_top1": pick(gap, "dinov2_minus_resnet50", head=head, metric="top1_endorsement"),
        "temperature": round(float(CAL[head]["temperature"]["temperature"]), 4),
        "log_loss_change_temperature": pick(gains, "change_vs_uncalibrated", head=head, calibrator="temperature", metric="log_loss"),
        "disagreement_full_coverage": pick(selective, "panel_disagreement", head=head, calibrator="identity", target_coverage=1.0),
        "disagreement_80pct_coverage": pick(selective, "panel_disagreement", head=head, calibrator="identity", target_coverage=0.8)}
write_json(p["reports"] / "foundation_dinov2_summary.json", verdict)
print(json.dumps(verdict, indent=2))